In [ ]:
!pip install torch_geometric transformers

### LIBRARIES

In [ ]:
import json
import os
import glob
import random
import numpy as np
from collections import Counter
import torch
import torch.nn.functional as F
from torch_geometric.data import Data, InMemoryDataset, HeteroData
from torch_geometric.loader import DataLoader
from torch_geometric.nn import RGCNConv, GATConv, global_mean_pool, HGTConv, Linear
from sklearn.metrics import f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

### CONFIGURATIONS

In [ ]:
JSON_DIR = "/kaggle/input/inter-doc-graphs/inter_doc_graphs"
OUT_DIR = "/kaggle/working/graphs/"
DATA_DIR = "/kaggle/working/graphs/"

os.makedirs(OUT_DIR, exist_ok=True)

ALLOWED_LABELS = {"Claim", "Premise", "Opposition"}
ALLOWED_EDGE_RELATIONS = {"follows", "green_support", "red_opposes", "inter_citation"}

# Batch processing configuration
USE_BATCH_PROCESSING = True
EMBEDDING_BATCH_SIZE = 32

# Model configuration
NUM_CLASSES = 3
RELATION_MAP = {"follows":0, "green_support":1, "red_opposes":2, "inter_citation":3}
NUM_RELATIONS = len(RELATION_MAP)

# Training hyperparameters
BATCH_SIZE = 4
LR = 1e-3
EPOCHS = 20
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SSL_WEIGHT = 0.1

# use float16 for embeddings to halve .pt size
USE_FLOAT16_STORAGE = True

random.seed(42)
torch.manual_seed(42)
np.random.seed(42)

tokenizer = AutoTokenizer.from_pretrained("law-ai/InCaseLawBERT")
model = AutoModel.from_pretrained("law-ai/InCaseLawBERT")
model.eval()

json_files = sorted(glob.glob(os.path.join(JSON_DIR, "*.json")))
print(f"✅ Found {len(json_files)} JSON files in {JSON_DIR}")

### CONVERT JSON TO PyG OBJECT

In [ ]:
def json_to_pyg(json_path, tokenizer, model, device='cpu'):
    """
    Convert a single graph JSON to PyTorch Geometric Data object.
    For single graph processing.
    """
    with open(json_path, "r") as f:
        js = json.load(f)

    nodes = js["nodes"]
    edges = js["edges"]

    # Node id mapping
    id2idx = {n["id"]: i for i, n in enumerate(nodes)}

    # Embeddings
    texts = [n["text"] for n in nodes]
    with torch.no_grad():
        tokens = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)
        out = model(**tokens)
        x = out.last_hidden_state[:, 0, :].cpu()  # CLS token

    # Labels
    label_map = {"Premise": 0, "Opposition": 1, "Claim": 2}
    y = torch.tensor([label_map[n["label"]] for n in nodes], dtype=torch.long)

    # Train mask (external nodes have train_mask=False)
    train_mask = torch.tensor([not n.get("external", False) for n in nodes], dtype=torch.bool)

    # Edges
    rel_map = {"follows": 0, "green_support": 1, "red_opposes": 2, "inter_citation": 3}
    src, tgt, etype = [], [], []
    for e in edges:
        if e["relation"] not in rel_map:
            continue
        src.append(id2idx[e["source"]])
        tgt.append(id2idx[e["target"]])
        etype.append(rel_map[e["relation"]])

    edge_index = torch.tensor([src, tgt], dtype=torch.long)
    edge_type = torch.tensor(etype, dtype=torch.long)

    # Node types for HGT
    node_type = torch.zeros(len(nodes), dtype=torch.long)

    # Store original embeddings for self-supervised learning
    original_embeddings = x.clone()

    return Data(
        x=x,
        y=y,
        edge_index=edge_index,
        edge_type=edge_type,
        train_mask=train_mask,
        node_type=node_type,
        original_embeddings=original_embeddings,
        doc_id=js.get("doc_id", os.path.basename(json_path))
    )


def batch_json_to_pyg(json_paths, tokenizer, model, device='cuda', batch_size=32):
    """
    Process multiple graphs in batch for embeddings for faster processing.
    """
    model = model.to(device)
    model.eval()

    # Collect all graph data and texts
    graph_data = []
    all_texts = []
    graph_text_ranges = []

    print(f"📖 Loading {len(json_paths)} graphs...")
    for json_path in json_paths:
        with open(json_path, "r", encoding="utf-8") as f:
            js = json.load(f)

        nodes = js["nodes"]
        edges = js["edges"]

        # Store graph metadata
        graph_data.append({
            "nodes": nodes,
            "edges": edges,
            "doc_id": js.get("doc_id", os.path.basename(json_path)),
            "json_path": json_path
        })

        # Collect texts for this graph
        texts = [n["text"] for n in nodes]
        start_idx = len(all_texts)
        all_texts.extend(texts)
        end_idx = len(all_texts)
        graph_text_ranges.append((start_idx, end_idx))

    print(f"📝 Total texts to embed: {len(all_texts)}")
    print(f"⚙️  Computing embeddings in batches of {batch_size}...")

    # Batch embed all texts
    all_embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(all_texts), batch_size), desc="Embedding"):
            batch_texts = all_texts[i:i+batch_size]
            tokens = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=128,
                return_tensors="pt"
            ).to(device)

            out = model(**tokens)
            embeddings = out.last_hidden_state[:, 0, :].cpu()  # CLS token
            all_embeddings.append(embeddings)

    # Concatenate all embeddings
    all_embeddings = torch.cat(all_embeddings, dim=0)
    print(f"✅ Computed {all_embeddings.shape[0]} embeddings")

    # Process each graph with its embeddings
    results = []
    label_map = {"Premise": 0, "Opposition": 1, "Claim": 2}
    rel_map = {"follows": 0, "green_support": 1, "red_opposes": 2, "inter_citation": 3}

    print(f"🔗 Building graph structures...")
    for idx, (graph_info, (start_idx, end_idx)) in enumerate(zip(graph_data, graph_text_ranges)):
        nodes = graph_info["nodes"]
        edges = graph_info["edges"]

        # Get embeddings for this graph
        x = all_embeddings[start_idx:end_idx]

        # Node id mapping
        id2idx = {n["id"]: i for i, n in enumerate(nodes)}

        # Labels
        y = torch.tensor([label_map[n["label"]] for n in nodes], dtype=torch.long)

        # Train mask (external nodes have train_mask=False)
        train_mask = torch.tensor([not n.get("external", False) for n in nodes], dtype=torch.bool)

        # Edges
        src, tgt, etype = [], [], []
        for e in edges:
            if e["relation"] not in rel_map:
                continue
            src.append(id2idx[e["source"]])
            tgt.append(id2idx[e["target"]])
            etype.append(rel_map[e["relation"]])

        edge_index = torch.tensor([src, tgt], dtype=torch.long)
        edge_type = torch.tensor(etype, dtype=torch.long)

        # Node types for HGT
        node_type = torch.zeros(len(nodes), dtype=torch.long)

        # Store original embeddings for self-supervised learning
        original_embeddings = x.clone()

        data = Data(
            x=x,
            y=y,
            edge_index=edge_index,
            edge_type=edge_type,
            train_mask=train_mask,
            node_type=node_type,
            original_embeddings=original_embeddings,
            doc_id=graph_info["doc_id"]
        )

        results.append(data)

    print(f"✅ Processed {len(results)} graphs")
    return results

### PyG OBJECT VERIFICATION

In [ ]:
def verify_pyg_data(data: Data):
    """Validate PyTorch Geometric Data object."""
    ok = True

    if data.x is None or data.x.dim() != 2:
        print("❌ Invalid node features")
        ok = False

    if data.y is None or data.y.size(0) != data.x.size(0):
        print("❌ Label / node mismatch")
        ok = False

    if data.edge_index.size(0) != 2:
        print("❌ edge_index must be [2, E]")
        ok = False

    if not hasattr(data, "edge_type"):
        print("❌ Missing edge_type")
        ok = False
    elif data.edge_type.size(0) != data.edge_index.size(1):
        print("❌ edge_type / edge_index mismatch")
        ok = False

    if data.edge_index.size(1) > 0 and data.edge_index.max() >= data.num_nodes:
        print("❌ Edge index out of bounds")
        ok = False

    return ok

### DELETE EXISTING FILES FOR SAFETY

In [ ]:
# # Delete old .pt files if they exist
# pt_files = glob.glob(os.path.join(OUT_DIR, "*.pt"))

# if len(pt_files) > 0:
#     print(f"🗑️  Found {len(pt_files)} old .pt files in {OUT_DIR}")
#     print("   Deleting old files...")
#     for f in pt_files:
#         os.remove(f)
#     print(f"✅ Deleted {len(pt_files)} old .pt files")
#     print("   → Now run Cell 6 to generate new .pt files with external nodes and original_embeddings")
# else:
#     print(f"✅ No old .pt files found in {OUT_DIR}")
#     print("   → Ready to generate new .pt files")

### SAVE PyG OBJECT

In [ ]:
# Paths for streaming mode
EMBEDDINGS_FILE = os.path.join(OUT_DIR, "all_embeddings.pt")
GRAPH_INDEX_FILE = os.path.join(OUT_DIR, "graph_index.pt")

def save_graph_tensor(data, out_path, use_float16=True):
    """Save PyG Data object to .pt file. use_float16=True halves embedding size for full corpus."""
    x = data.x
    oe = getattr(data, "original_embeddings", None)
    if use_float16 and x is not None:
        x = x.half()
    if use_float16 and oe is not None:
        oe = oe.half()
    torch.save({
        "x": x,
        "y": data.y,
        "edge_index": data.edge_index,
        "edge_type": data.edge_type,
        "train_mask": data.train_mask,
        "node_type": getattr(data, "node_type", None),
        "original_embeddings": oe,
        "doc_id": getattr(data, "doc_id", None)
    }, out_path)

### CONVERT JSON TO PYG IN BATCH

In [ ]:
if USE_BATCH_PROCESSING:
    print("="*80)
    print("BATCH PROCESSING MODE")
    print("="*80)
    print(f"Device: {DEVICE}")
    print(f"Embedding batch size: {EMBEDDING_BATCH_SIZE}")

    # Process all graphs in batch
    try:
        all_data = batch_json_to_pyg(
            json_files,
            tokenizer,
            model,
            device=DEVICE,
            batch_size=EMBEDDING_BATCH_SIZE
        )

        # Validate and save
        bad_graphs = []
        external_nodes_saved = 0
        has_original_emb_saved = 0
        total_nodes_saved = 0


        for data, json_file in zip(all_data, json_files):
            if not verify_pyg_data(data):
                print(f"❌ PyG validation failed: {json_file}")
                bad_graphs.append(json_file)
                continue

            # Check if this graph has external nodes and original_embeddings
            external_mask = ~data.train_mask
            external_nodes_saved += external_mask.sum().item()
            total_nodes_saved += data.train_mask.size(0)
            if hasattr(data, 'original_embeddings') and data.original_embeddings is not None:
                has_original_emb_saved += 1

            out_path = os.path.join(
                OUT_DIR,
                os.path.basename(json_file).replace(".json", ".pt")
            )
            save_graph_tensor(data, out_path, use_float16=USE_FLOAT16_STORAGE)


        print(f"\n✅ Processed {len(json_files) - len(bad_graphs)}/{len(json_files)} graphs")
        print(f"❌ Bad graphs: {len(bad_graphs)}")
        print(f"\n📊 Summary of saved data:")
        print(f"   Total nodes saved: {total_nodes_saved}")
        print(f"   External nodes saved: {external_nodes_saved} ({100*external_nodes_saved/max(total_nodes_saved,1):.1f}%)")
        print(f"   Graphs with original_embeddings: {has_original_emb_saved}/{len(json_files) - len(bad_graphs)}")

        if external_nodes_saved == 0:
            print("\n⚠️  WARNING: No external nodes were saved!")
            print("   → Check if JSON files actually have external nodes")
            print("   → Verify JSON_DIR points to inter-doc-graphs/ folder")
        elif has_original_emb_saved < len(json_files) - len(bad_graphs):
            print("\n⚠️  WARNING: Some graphs missing original_embeddings!")
        else:
            print("\n✅ All graphs saved with external nodes and original_embeddings!")

    except Exception as e:
        print(f"❌ Batch processing failed: {e}")
        import traceback
        traceback.print_exc()
        print("\nFalling back to single-graph processing...")
        USE_BATCH_PROCESSING = False

# Fallback to single-graph processing if batch processing fails
if not USE_BATCH_PROCESSING:
    print("\n" + "="*80)
    print("SINGLE-GRAPH PROCESSING MODE")
    print("="*80)

    bad_graphs = []
    for jf in tqdm(json_files[:10], desc="Processing graphs"):
        data = json_to_pyg(jf, tokenizer, model, device=DEVICE)

        if not verify_pyg_data(data):
            print("❌ PyG validation failed:", jf)
            bad_graphs.append(jf)
            continue

        out_path = os.path.join(
            OUT_DIR,
            os.path.basename(jf).replace(".json", ".pt")
        )
        save_graph_tensor(data, out_path, use_float16=USE_FLOAT16_STORAGE)

    print(f"✅ Processed {len(json_files) - len(bad_graphs)}/{len(json_files)} graphs")
    print(f"❌ Bad graphs: {len(bad_graphs)}")

### DATASET CONVERTION

In [ ]:
class ClausesDataset(torch.utils.data.Dataset):
    def __init__(self, root):
        self.files = sorted([f for f in glob.glob(os.path.join(root, "*.pt")) if not f.endswith("all_embeddings.pt") and "graph_index" not in f])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        d = torch.load(self.files[idx], weights_only=False)
        x = d["x"]
        oe = d.get("original_embeddings")
        if x is not None and x.dtype == torch.float16:
            x = x.float()
        if oe is not None and oe.dtype == torch.float16:
            oe = oe.float()
        return Data(
            x=x,
            y=d["y"],
            edge_index=d["edge_index"],
            edge_type=d["edge_type"],
            train_mask=d.get("train_mask"),
            node_type=d.get("node_type"),
            original_embeddings=oe,
            doc_id=d.get("doc_id")
        )

### RELATIONAL GRAPH CONVOLUTIONAL NETWORK MODEL 

In [ ]:
class RGCNModel(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim=256, out_dim=NUM_CLASSES, num_relations=NUM_RELATIONS, num_layers=2, dropout=0.2, use_ssl=True):
        super().__init__()
        self.convs = torch.nn.ModuleList()
        self.convs.append(RGCNConv(in_dim, hidden_dim, num_relations, num_bases=30))
        for _ in range(num_layers - 1):
            self.convs.append(RGCNConv(hidden_dim, hidden_dim, num_relations, num_bases=30))
        self.lin = torch.nn.Linear(hidden_dim, out_dim)
        self.dropout = dropout
        self.use_ssl = use_ssl

        # SSL: Reconstruction head for external nodes
        if self.use_ssl:
            self.reconstruction_head = torch.nn.Sequential(
                torch.nn.Linear(hidden_dim, hidden_dim),
                torch.nn.ReLU(),
                torch.nn.Linear(hidden_dim, in_dim)
            )

    def forward(self, data, return_embeddings=False):
        x, edge_index, edge_type = data.x, data.edge_index, data.edge_type
        for conv in self.convs:
            x = conv(x, edge_index, edge_type)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)

        # Store embeddings before classification
        node_embeddings = x if return_embeddings or self.use_ssl else None

        out = self.lin(x)

        if return_embeddings:
            return out, node_embeddings
        return out

    def compute_reconstruction_loss(self, embeddings, original_embeddings, external_mask):
        """Compute self-supervised reconstruction loss for external nodes."""
        if not self.use_ssl or external_mask.sum() == 0:
            return torch.tensor(0.0, device=embeddings.device)

        # Reconstruct original embeddings for external nodes
        reconstructed = self.reconstruction_head(embeddings[external_mask])
        targets = original_embeddings[external_mask]

        # MSE loss: learn to reconstruct original embeddings
        loss = F.mse_loss(reconstructed, targets, reduction='mean')
        return loss

### RELATIONAL GRAPH ATTENTION NETWORKS MODEL

In [ ]:
class RelationalGATModel(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim=256, out_dim=NUM_CLASSES, num_relations=NUM_RELATIONS, num_layers=2, heads=4, dropout=0.2, use_ssl=True):
        super().__init__()
        self.num_relations = num_relations
        self.rel_convs = torch.nn.ModuleList([
            GATConv(in_dim, hidden_dim // heads, heads=heads, concat=True,
                   dropout=dropout, add_self_loops=False)
            for _ in range(num_relations)
        ])
        self.middle_convs = torch.nn.ModuleList([
            GATConv(hidden_dim, hidden_dim // heads, heads=heads, concat=True,
                   dropout=dropout, add_self_loops=False)
            for _ in range(num_layers - 1)
        ])
        self.lin = torch.nn.Linear(hidden_dim, out_dim)
        self.dropout = dropout
        self.use_ssl = use_ssl

        # SSL: Reconstruction head for external nodes
        if self.use_ssl:
            self.reconstruction_head = torch.nn.Sequential(
                torch.nn.Linear(hidden_dim, hidden_dim),
                torch.nn.ReLU(),
                torch.nn.Linear(hidden_dim, in_dim)
            )

    def forward(self, data, return_embeddings=False):
        x, edge_index, edge_type = data.x, data.edge_index, data.edge_type
        h_list = []
        for r in range(self.num_relations):
            mask = (edge_type == r)
            if mask.sum() > 0:
                rel_edges = edge_index[:, mask]
                h_r = self.rel_convs[r](x, rel_edges)
                h_list.append(h_r)
        if len(h_list) > 0:
            h = torch.stack(h_list, dim=0).mean(dim=0)
        else:
            h = x
        for conv in self.middle_convs:
            h = conv(h, edge_index)
            h = F.elu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)

        # Store embeddings before classification
        node_embeddings = h if return_embeddings or self.use_ssl else None

        out = self.lin(h)

        if return_embeddings:
            return out, node_embeddings
        return out

    def compute_reconstruction_loss(self, embeddings, original_embeddings, external_mask):
        """Compute self-supervised reconstruction loss for external nodes."""
        if not self.use_ssl or external_mask.sum() == 0:
            return torch.tensor(0.0, device=embeddings.device)

        # Reconstruct original embeddings for external nodes
        reconstructed = self.reconstruction_head(embeddings[external_mask])
        targets = original_embeddings[external_mask]

        # MSE loss: learn to reconstruct original embeddings
        loss = F.mse_loss(reconstructed, targets, reduction='mean')
        return loss

### HETEROGENEOUS GRAPH TRANSFORMER MODEL

In [ ]:
class HGTModel(torch.nn.Module):
    """
    Heterogeneous Graph Transformer.
    Uses edge types (relations) for heterogeneity.
    """
    def __init__(self, in_dim, hidden_dim=256, out_dim=NUM_CLASSES,
                 num_node_types=1, num_edge_types=NUM_RELATIONS,
                 num_heads=4, num_layers=2, dropout=0.2, use_ssl=True):
        super().__init__()
        self.num_node_types = 1
        self.num_edge_types = num_edge_types
        self.hidden_dim = hidden_dim
        self.use_ssl = use_ssl

        self.node_type_names = ["node"]

        edge_type_names = ["follows", "green_support", "red_opposes", "inter_citation"]
        self.edge_types = [
            (self.node_type_names[0], edge_type_names[i], self.node_type_names[0])
            for i in range(num_edge_types)
        ]

        self.metadata = (self.node_type_names, self.edge_types)

        self.node_type_emb = torch.nn.Embedding(1, in_dim)

        # HGT layers
        self.convs = torch.nn.ModuleList()
        self.convs.append(HGTConv(in_dim, hidden_dim, self.metadata, num_heads))
        for _ in range(num_layers - 1):
            self.convs.append(HGTConv(hidden_dim, hidden_dim, self.metadata, num_heads))

        # Final classification layer
        self.lin = Linear(hidden_dim, out_dim)

        # Self-supervised learning: Reconstruction head for external nodes
        if self.use_ssl:
            self.reconstruction_head = torch.nn.Sequential(
                Linear(hidden_dim, hidden_dim),
                torch.nn.ReLU(),
                Linear(hidden_dim, in_dim)
            )

        self.dropout = dropout

    def forward(self, data, return_embeddings=False):
        """Forward pass with optional self-supervised learning."""
        node_type = torch.zeros(data.x.size(0), dtype=torch.long, device=data.x.device)

        edge_type = data.edge_type
        edge_index = data.edge_index

        original_embeddings = getattr(data, 'original_embeddings', None)
        if original_embeddings is None and self.use_ssl:
            original_embeddings = data.x.clone()

        # Use minimal node type embedding
        x = data.x + self.node_type_emb(node_type) * 0.1

        x_dict = {self.node_type_names[0]: x}
        edge_index_dict = {}

        # Group edges by type
        for et in range(self.num_edge_types):
            mask = (edge_type == et)
            if mask.sum() > 0:
                edge_key = self.edge_types[et]
                edge_index_dict[edge_key] = edge_index[:, mask]

        # HGT message passing
        for conv in self.convs:
            x_dict = conv(x_dict, edge_index_dict)
            if x_dict[self.node_type_names[0]] is not None:
                x = x_dict[self.node_type_names[0]]
                x = F.relu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)
                x_dict[self.node_type_names[0]] = x
            else:
                x = data.x

        # Store embeddings before classification
        node_embeddings = x if return_embeddings or self.use_ssl else None

        # Final classification
        out = self.lin(x)

        if return_embeddings:
            return out, node_embeddings
        return out

    def compute_reconstruction_loss(self, embeddings, original_embeddings, external_mask):
        """Compute self-supervised reconstruction loss for external nodes."""
        if not self.use_ssl or external_mask.sum() == 0:
            return torch.tensor(0.0, device=embeddings.device)

        # Reconstruct original embeddings for external nodes
        reconstructed = self.reconstruction_head(embeddings[external_mask])
        targets = original_embeddings[external_mask]

        # MSE loss: learn to reconstruct original embeddings
        loss = F.mse_loss(reconstructed, targets, reduction='mean')
        return loss

### CHECK FOR DATA VALIDITY

In [ ]:
print("="*80)
print("CONFIGURATION CHECK - Training Data")
print("="*80)
print(f"DATA_DIR: {DATA_DIR}")

if not os.path.exists(DATA_DIR):
    print(f"❌ ERROR: DATA_DIR does not exist: {DATA_DIR}")
    print("   Please update DATA_DIR to match OUT_DIR from Cell 2")
    raise FileNotFoundError(f"DATA_DIR not found: {DATA_DIR}")

pt_files = glob.glob(os.path.join(DATA_DIR, "*.pt"))
print(f"✅ Found {len(pt_files)} .pt files in {DATA_DIR}")

if len(pt_files) == 0:
    print("⚠️  WARNING: No .pt files found!")
    print("   → Run Cell 6 first to generate .pt files")
else:
    # Check ALL .pt files before finalizing external-node / SSL status
    total_external = 0
    total_nodes = 0
    graphs_with_original_emb = 0
    scan_ok = False
    try:
        for pt_path in pt_files:
            d = torch.load(pt_path, weights_only=True)
            if "train_mask" in d:
                ext = (~d["train_mask"]).sum().item()
                n = d["train_mask"].size(0)
                total_external += ext
                total_nodes += n
            if d.get("original_embeddings") is not None:
                graphs_with_original_emb += 1
        scan_ok = True
    except TypeError:
        total_external, total_nodes, graphs_with_original_emb = 0, 0, 0
        for pt_path in pt_files:
            d = torch.load(pt_path)
            if "train_mask" in d:
                ext = (~d["train_mask"]).sum().item()
                n = d["train_mask"].size(0)
                total_external += ext
                total_nodes += n
            if d.get("original_embeddings") is not None:
                graphs_with_original_emb += 1
        scan_ok = True
    except Exception as e:
        print(f"   ⚠️  Error scanning .pt files: {e}")
    if scan_ok:
        pct = 100 * total_external / max(total_nodes, 1)
        print(f"   All {len(pt_files)} graphs: {total_external} external nodes in {total_nodes} nodes ({pct:.1f}%)")
        print(f"   original_embeddings: {graphs_with_original_emb}/{len(pt_files)} graphs")
        if total_external == 0:
            print("   ⚠️  WARNING: No external nodes in any graph!")
            print("   → Re-run Cell 5 (delete) and Cell 6 (regenerate) if you expect inter-doc nodes")
        elif graphs_with_original_emb < len(pt_files):
            print("   ⚠️  WARNING: Some graphs missing original_embeddings!")
            print("   → Re-run Cell 6 to regenerate")
        else:
            print("   ✅ Data ready for SSL (external nodes + original_embeddings present)")
print("="*80)
print("")

### LOAD DATASET 

In [ ]:
dataset = ClausesDataset(DATA_DIR)
print(f"📊 Loaded {len(dataset)} graphs from {DATA_DIR}")

# Train/Val/Test split (80/10/10)
indices = list(range(len(dataset)))
random.shuffle(indices)
train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))

train_indices = indices[:train_size]
val_indices = indices[train_size:train_size+val_size]
test_indices = indices[train_size+val_size:]

train_dataset = torch.utils.data.Subset(dataset, train_indices)
val_dataset = torch.utils.data.Subset(dataset, val_indices)
test_dataset = torch.utils.data.Subset(dataset, test_indices)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"📊 Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

# Get feature dimension
pt_files = sorted([f for f in glob.glob(os.path.join(DATA_DIR, "*.pt")) if "all_embeddings" not in f and "graph_index" not in f])
sample_dict = torch.load(pt_files[0], weights_only=False)
in_dim = sample_dict["x"].shape[1]
print(f"📐 Feature dimension: {in_dim}")

### EVALUATION OF MODEL

In [ ]:
def evaluate_node_classification(model, loader, device=DEVICE):
    """Evaluate model - ONLY on non-external nodes (train_mask=True)."""
    model.eval()
    ys, preds = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch)

            # Only evaluate on non-external nodes
            eval_mask = batch.train_mask  # train_mask = not external
            if eval_mask.sum() == 0:
                continue

            pred = out[eval_mask].argmax(dim=-1).cpu().numpy()
            y = batch.y[eval_mask].cpu().numpy()
            preds.append(pred)
            ys.append(y)

    if len(ys) == 0:
        return {"acc": 0.0, "macro_f1": 0.0}

    y_true = np.concatenate(ys)
    y_pred = np.concatenate(preds)

    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "per_class_f1": f1_score(y_true, y_pred, average=None).tolist()
    }


class FocalLoss(torch.nn.Module):
    """
    Focal Loss for addressing class imbalance.
    Focuses learning on hard examples.
    """
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha  # Class weights
        self.gamma = gamma  # Focusing parameter
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss

### TRAIN MODEL FUNCTION

In [ ]:
def train_model(model, train_loader, val_loader, model_name,
                         use_ssl=False, ssl_weight=0.1,
                         dropout=0.4, weight_decay=1e-4,
                         hidden_dim=128, use_focal_loss=False):
    """
    Training function with:
    - Stronger regularization
    - Focal loss option
    - Learning rate scheduling
    - Early stopping
    """
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=weight_decay)

    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

    # Focal Loss function
    if use_focal_loss:
        all_labels = []
        for batch in train_loader:
            eval_mask = batch.train_mask
            if eval_mask.sum() > 0:
                all_labels.extend(batch.y[eval_mask].cpu().numpy().tolist())

        if len(all_labels) > 0:
            unique_classes = np.unique(all_labels)
            computed_weights = compute_class_weight('balanced', classes=unique_classes, y=all_labels)
            alpha = torch.ones(NUM_CLASSES, dtype=torch.float).to(DEVICE)
            for i, cls in enumerate(unique_classes):
                alpha[int(cls)] = float(computed_weights[i])
        else:
            alpha = torch.ones(NUM_CLASSES).to(DEVICE)

        criterion = FocalLoss(alpha=alpha, gamma=2.0)
        print("📊 Using Focal Loss for class imbalance")
    else:
        # Standard weighted cross-entropy
        all_labels = []
        for batch in train_loader:
            eval_mask = batch.train_mask
            if eval_mask.sum() > 0:
                all_labels.extend(batch.y[eval_mask].cpu().numpy().tolist())

        if len(all_labels) > 0:
            unique_classes = np.unique(all_labels)
            computed_weights = compute_class_weight('balanced', classes=unique_classes, y=all_labels)
            class_weights = torch.ones(NUM_CLASSES, dtype=torch.float).to(DEVICE)
            for i, cls in enumerate(unique_classes):
                class_weights[int(cls)] = float(computed_weights[i])
        else:
            class_weights = torch.ones(NUM_CLASSES).to(DEVICE)

        criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
        print(f"📊 Class weights: {class_weights.cpu().numpy()}")

    if use_ssl:
        print(f"🔬 Self-supervised learning ENABLED (SSL weight: {ssl_weight})")
    else:
        print(f"⚠️  Self-supervised learning DISABLED")

    best_val_f1 = 0.0
    best_model_state = None
    patience = 5
    patience_counter = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        total_class_loss = 0.0
        total_ssl_loss = 0.0
        num_batches = 0

        for batch in train_loader:
            batch = batch.to(DEVICE)
            optimizer.zero_grad()

            # Forward pass
            if use_ssl and hasattr(model, 'compute_reconstruction_loss'):
                out, node_embeddings = model(batch, return_embeddings=True)
            else:
                out = model(batch)
                node_embeddings = None

            # Classification loss
            eval_mask = batch.train_mask
            if eval_mask.sum() > 0:
                classification_loss = criterion(out[eval_mask], batch.y[eval_mask])
            else:
                classification_loss = torch.tensor(0.0, device=DEVICE)

            # SSL loss
            ssl_loss = torch.tensor(0.0, device=DEVICE)
            if use_ssl and hasattr(batch, 'original_embeddings'):
                if hasattr(batch, 'original_embeddings') and batch.original_embeddings is not None:
                    external_mask = ~batch.train_mask
                    if external_mask.sum() > 0 and node_embeddings is not None:
                        ssl_loss = model.compute_reconstruction_loss(
                            node_embeddings, batch.original_embeddings, external_mask
                        )

            # Total loss with gradient clipping
            total_loss_batch = classification_loss + ssl_weight * ssl_loss
            total_loss_batch.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Gradient clipping
            optimizer.step()

            total_loss += total_loss_batch.item()
            total_class_loss += classification_loss.item()
            total_ssl_loss += ssl_loss.item()
            num_batches += 1

        # Validation
        val_metrics = evaluate_node_classification(model, val_loader, DEVICE)
        scheduler.step()

        # Logging
        avg_loss = total_loss / max(num_batches, 1)
        avg_class_loss = total_class_loss / max(num_batches, 1)
        avg_ssl_loss = total_ssl_loss / max(num_batches, 1)
        current_lr = scheduler.get_last_lr()[0]

        if use_ssl:
            print(f"Epoch {epoch:03d} | total_loss {avg_loss:.4f} | class_loss {avg_class_loss:.4f} | ssl_loss {avg_ssl_loss:.4f} | val_acc {val_metrics['acc']:.4f} | val_macroF1 {val_metrics['macro_f1']:.4f} | lr {current_lr:.6f}")
        else:
            print(f"Epoch {epoch:03d} | loss {avg_loss:.4f} | val_acc {val_metrics['acc']:.4f} | val_macroF1 {val_metrics['macro_f1']:.4f} | lr {current_lr:.6f}")

        # Early stopping
        if val_metrics['macro_f1'] > best_val_f1:
            best_val_f1 = val_metrics['macro_f1']
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"⚠️  Early stopping at epoch {epoch} (no improvement for {patience} epochs)")
                break

    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"✅ Loaded best model (val_macroF1: {best_val_f1:.4f})")

    return model, val_metrics


### TRAIN R-GCN MODEL

In [ ]:
results = {}

print("\n" + "="*80)
print("Training R-GCN Model")
print("="*80)
rgcn = RGCNModel(in_dim=in_dim, num_relations=NUM_RELATIONS)
rgcn, val_metrics = train_model(rgcn, train_loader, val_loader, "R-GCN", use_ssl=True)
test_metrics = evaluate_node_classification(rgcn, test_loader, DEVICE)
results['R-GCN'] = test_metrics
print("\n" + "="*80)



### TRAIN R-GAT MODEL

In [ ]:
print("\n" + "="*80)
print("Training Relational-GAT Model")
print("="*80)
rgat = RelationalGATModel(in_dim=in_dim, num_relations=NUM_RELATIONS)
rgat, val_metrics = train_model(rgat, train_loader, val_loader, "Relational-GAT", use_ssl=True)
test_metrics = evaluate_node_classification(rgat, test_loader, DEVICE)
results['Relational-GAT'] = test_metrics
print("\n" + "="*80)

### TRAIN HGT MODEL

In [ ]:
print("\n" + "="*80)
print("Training HGT Model (Heterogeneous Graph Transformer)")
print("="*80)
hgt = HGTModel(in_dim=in_dim, num_node_types=1, num_edge_types=NUM_RELATIONS, use_ssl=True)
hgt, val_metrics = train_model(hgt, train_loader, val_loader, "HGT", use_ssl=True, ssl_weight=SSL_WEIGHT)
test_metrics = evaluate_node_classification(hgt, test_loader, DEVICE)
results['HGT'] = test_metrics
print("\n" + "="*80)

### SUMMARY OF THREE MODELS

In [ ]:
print("\n" + "="*80)
print("Summary - All Models")
print("="*80)
for model_name, metrics in results.items():
    print(f"\n{model_name:15s}: Acc={metrics['acc']:.4f}, F1={metrics['macro_f1']:.4f}")
    print(f"\t\t Per-class F1: {metrics['per_class_f1']}")

### SAVE MODEL

In [ ]:
# Save best model
torch.save({
    "model_state": rgcn.state_dict(),
    "model_type": "rgcn",
    "config": {
        "in_dim": in_dim,
        "hidden_dim": 256,
        "out_dim": NUM_CLASSES,
        "num_relations": NUM_RELATIONS,
        "num_layers": 2,
        "dropout": 0.2,
    }
}, "rgcn_encoder_final.pt")
print("RGCN encoder saved successfully!")